# Visual Intelligence — End-to-End Colab Demo

This notebook clones the `codex/end-to-end-pipeline` branch, installs dependencies, runs the dependency-free tests, and executes a two-item dataset containing included present-person and absent-person videos. The first run downloads several gigabytes of model weights and may take a few minutes.

Before selecting **Runtime → Run all**, select **Runtime → Change runtime type → GPU**. A T4 is sufficient for this demo; no manual file uploads are required.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU with Runtime → Change runtime type → GPU, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import os
import subprocess

repo_dir = Path("/content/Visual-Intelligence")
branch = "codex/end-to-end-pipeline"
repo_url = "https://github.com/SamhithKakarla/Visual-Intelligence.git"

if not (repo_dir / ".git").exists():
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repo_url, str(repo_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "switch", branch], check=True)
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)

os.chdir(repo_dir)
print("Working directory:", Path.cwd())
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["ls", "-lh", "test_image.jpg", "test_video.mp4", "test_video_absent.mp4"], check=True)

## Install dependencies

The first run installs FFmpeg and Python packages. It will also download YOLO, InsightFace, and Qwen weights later during inference.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install --upgrade -q pip setuptools wheel
!python -m pip install -q -r requirements.txt

## Validate the control flow

In [ ]:
!python -m unittest discover -s tests -v

## Build and run the two-item dataset

Both pairs are processed in one batch. The present case should continue to Phase 2; the absent case should stop after Phase 1. Phase 2 receives up to 48 target-highlighted evidence frames per appearance.

In [ ]:
import json
from pathlib import Path

from visual_intelligence.batch import run_batch
from visual_intelligence.config import Phase1Config, Phase2Config, PipelineConfig

manifest = {
    "dataset_id": "two-video-smoke-test",
    "items": [
        {
            "case_id": "known-present",
            "reference_image": "test_image.jpg",
            "reference_video": "test_video.mp4",
        },
        {
            "case_id": "known-absent",
            "reference_image": "test_image.jpg",
            "reference_video": "test_video_absent.mp4",
        },
    ],
}
manifest_path = Path("demo_dataset.json")
manifest_path.write_text(json.dumps(manifest, indent=2))

config = PipelineConfig(
    phase1=Phase1Config(
        search_fps=2.0,
        identity_threshold=0.4,
        max_evidence_frames=48,
    ),
    phase2=Phase2Config(model_id="Qwen/Qwen3-VL-2B-Instruct"),
    runs_dir=Path("/content/visual_intelligence_runs"),
    keep_artifacts=True,
)

batch_result = run_batch(
    manifest_path=manifest_path,
    output_path="batch_results.json",
    config=config,
)

print(json.dumps(batch_result, indent=2))

## Inspect the combined public and diagnostic outputs

In [ ]:
import json
from IPython.display import JSON, display
from PIL import Image

with open("batch_results.json") as stream:
    public_result = json.load(stream)
with open("batch_results.debug.json") as stream:
    debug_result = json.load(stream)

display(JSON(public_result))
display(JSON(debug_result))

print("\nGate verification:")
for item in debug_result["items"]:
    person_exists = item["phase1"]["person_exists"]
    phase2_called = item["phase2"] is not None
    print(f"{item['case_id']}: person_exists={person_exists}, phase2_called={phase2_called}")
    if person_exists and item["phase1"]["appearances"]:
        evidence_path = item["phase1"]["appearances"][0]["frames"][0]["context_frame_path"]
        display(Image.open(evidence_path))

absent_item = next(item for item in debug_result["items"] if item["case_id"] == "known-absent")
if not absent_item["phase1"]["person_exists"] and absent_item["phase2"] is None:
    print("PASS: known-absent exited after Phase 1 and never called Qwen.")
else:
    print("CHECK: known-absent produced a Phase 1 false positive at the current threshold.")

## Download the results

Running the next cell packages the manifest, combined outputs, and individual per-case outputs into one download.

In [ ]:
from google.colab import files

!zip -r -q visual_intelligence_outputs.zip demo_dataset.json batch_results.json batch_results.debug.json batch_results.items
files.download("visual_intelligence_outputs.zip")